In [ ]:
source("helpers/init.r")
source("helpers/oncoplot_utils.r")

In [ ]:
cohorts    <- fread(paste0(READY_DATA, "cohorts.txt"))
plt_cohort <- cohorts %>% filter(!ref) %>% pull(cohort) %>% unique()
ref_cohort <- cohorts %>% filter(ref)  %>% pull(cohort) %>% unique()

# 0 - Load data

In [ ]:
driver <- fread(paste0(BASE_DATA, "drivers.txt"))

In [ ]:
validate_columns(driver, c("sampleId", "gene", "type", "event",
                         "driver", "category"), "drivers")

# 1 - Rank genes

In [ ]:
ready <-
  cohorts %>%
  select(sampleId, cohort) %>%
  left_join(driver, by = "sampleId", relationship = "many-to-many")

ref <-
  ready %>%
  distinct(cohort, type, sampleId, gene) %>%
  group_by(cohort, type, gene) %>%
  summarise(ct = n(), .groups = "drop") %>%
  group_by(cohort, type) %>%
  mutate(rk = row_number(desc(ct))) %>%
  select(-ct) %>%
  ungroup()


# 2 - Build sample × gene grid

In [ ]:
ref_dt   <- as.data.table(ref)
ready_dt <- as.data.table(ready)
samps_dt <- unique(ready_dt[, .(sampleId, cohort)])

tall_ref_dt <- ref_dt[samps_dt, on = "cohort", allow.cartesian = TRUE]
events_dt   <- unique(ready_dt[, .(sampleId, cohort, gene, pathway, category,
                                   driver_simple, type, event)])

onco_base <- merge(tall_ref_dt, events_dt,
                   by = c("sampleId", "cohort", "gene", "type"), all.x = TRUE)
onco_base[is.na(driver_simple), driver_simple := "none"]
onco_base[is.na(category),      category      := "none"]
onco_base[is.na(event),         event         := "none"]
onco_base[is.na(pathway),       pathway       := "Other"]
setkeyv(onco_base, c("cohort", "sampleId", "type", "gene"))
onco_base <- as_tibble(unique(onco_base[!is.na(gene)]))


# 3 - Sort samples

# 4 - Prepare cohort data

In [ ]:
top_map <- list("GERMLINE" = ONCO_DISPLAY_TOP_GERMLINE, "SOMATIC" = ONCO_DISPLAY_TOP_SOMATIC)

cohort_data <- onco_base %>% filter(cohort == plt_cohort)

germline <- cohort_data %>%
  filter(type == "GERMLINE", rk <= top_map[["GERMLINE"]]) %>%
  group_by(sampleId, gene) %>% slice_head(n = 1) %>% ungroup()

somatic <- cohort_data %>%
  filter(type == "SOMATIC", rk <= top_map[["SOMATIC"]]) %>%
  group_by(sampleId, gene) %>% slice_head(n = 1) %>% ungroup()

oncogene_ready <- list(
  germline = if (nrow(germline) == 0) NA else order_oncoplot(germline),
  somatic  = order_oncoplot(somatic)
)

# 5 - Driver labels

In [ ]:
get_label_category <- function(base) {
  base %>%
    filter(category != "none") %>%
    group_by(gene, category) %>%
    summarise(ct = n(), .groups = "drop") %>%
    group_by(gene) %>%
    slice_max(ct, n = 1) %>%
    transmute(gene, field = "Type", val0 = NA,
              val = ifelse(category == "ONCO", "ONC", "TSG")) %>%
    ungroup()
}

add_pathway <- function(base) {
  counts <- base %>%
    group_by(gene, pathway) %>% summarise(ct = n(), .groups = "drop")
  # Always prefer a specific pathway over NULL/Other, regardless of count
  specific <- counts %>%
    filter(!pathway %in% c("NULL", "Other") & !is.na(pathway)) %>%
    group_by(gene) %>% slice_max(ct, n = 1, with_ties = FALSE) %>% ungroup()
  fallback <- counts %>%
    filter(!gene %in% specific$gene) %>%
    group_by(gene) %>% slice_max(ct, n = 1, with_ties = FALSE) %>% ungroup()
  pw <- bind_rows(specific, fallback) %>%
    mutate(val = ifelse(pathway %in% names(pathway_map), pathway_map[pathway], pathway)) %>%
    select(gene, val)
  data.frame(field = "Pathway", gene = unique(base$gene), val0 = NA) %>%
    left_join(pw, by = "gene")
}

get_labels <- function(type = "SOMATIC") {
  n_top <- top_map[[type]]
  base <- bind_rows(
    onco_base %>% filter(cohort == plt_cohort, type == !!type, rk <= n_top),
    onco_base %>% filter(cohort == ref_cohort,  type == !!type,
                         gene %in% (onco_base %>% filter(cohort == plt_cohort, type == !!type, rk <= n_top) %>% pull(gene)))
  )
  bind_rows(get_label_pcts(base, "gene", "none", "event"),
            get_label_category(base),
            get_highlights(base, "gene", "none", "event"),
            add_pathway(base)) %>%
    filter(!is.na(gene)) %>%
    mutate(field2 = factor(
      ifelse(!field %in% c("Pathway", "Type", "Direction"),
             paste0(field, " (%)"), field),
      levels = c("Pathway", "Type",
                 paste0(plt_cohort, " (%)"), paste0(ref_cohort, " (%)"), "Direction")))
}

driver_labels <- list(
  SOMATIC  = get_labels("SOMATIC"),
  GERMLINE = if (nrow(onco_base %>% filter(type == "GERMLINE")) > 0) get_labels("GERMLINE") else NA
)

# 6 - Oncoplot figures

In [ ]:
plot_oncodriver <- function(type = "somatic") {
  df <- oncogene_ready[[type]]
  if (!is.data.frame(df)) return(ggplot() + theme_void())
  pct_driver <- df %>% group_by(sampleId) %>%
    summarise(has = any(driver_simple != "none")) %>% pull(has) %>% mean()
  y_label <- if (type == "germline") "Germline Drivers" else "Somatic Drivers"
  ggplot(df %>% filter(driver_simple != "none"),
         aes(x = sampleId, y = gene, fill = event)) +
    geom_tile() +
    scale_fill_manual(values = driver_fill_map,
                      breaks = setdiff(names(driver_fill_map), "none")) +
    labs(x = paste0(df %>% filter(driver_simple != "none") %>%
                    pull(sampleId) %>% n_distinct(), " Patients"),
         y = y_label) +
    scale_x_discrete(position = "top") +
    onco_theme +
    guides(fill = guide_legend(nrow = 1))
}

onco_plts <- list(
  somatic  = plot_oncodriver("somatic"),
  germline = plot_oncodriver("germline")
)

# 7 - Driver label figures

In [ ]:
plot_driver_labels <- function(type = "SOMATIC") {
  df <- driver_labels[[type]]
  if (!is.data.frame(df)) return(ggplot() + theme_void())
  levels <- rev(df %>% filter(field == plt_cohort) %>% arrange(desc(val0)) %>% pull(gene))
  df <- df %>% mutate(gene = factor(gene, levels = levels))
  ggplot(df, aes(x = field2, y = gene, label = val)) +
    geom_text(size = 3) +
    scale_x_discrete(position = "top", guide = guide_axis(n.dodge = 2)) +
    label_theme
}

label_plts <- list(
  SOMATIC  = plot_driver_labels("SOMATIC"),
  GERMLINE = plot_driver_labels("GERMLINE")
)

In [ ]:
# ── Unit test: gene alignment between oncoplot and label panels ───────────────
check_gene_alignment <- function(type_onco, type_label) {
  genes_onco  <- levels(onco_plts[[type_onco]]$data$gene)
  genes_label <- unique(label_plts[[type_label]]$data$gene)
  missing_in_label <- setdiff(genes_onco, genes_label)
  missing_in_onco  <- setdiff(genes_label, genes_onco)
  if (length(missing_in_label) > 0 || length(missing_in_onco) > 0)
    stop(sprintf("Gene mismatch in %s/%s\n  missing in label: %s\n  missing in onco: %s",
                 type_onco, type_label,
                 paste(missing_in_label, collapse = ", "),
                 paste(missing_in_onco,  collapse = ", ")))
  message(sprintf("\u2713 %s/%s: %d genes aligned", type_onco, type_label, length(genes_onco)))
}

check_gene_alignment("somatic",  "SOMATIC")
check_gene_alignment("germline", "GERMLINE")

# 8 - Save figures

In [ ]:
# ── Unit test: add_pathway specific/fallback priority ────────────────────────
local({
  mock_counts <- data.frame(
    gene    = c("TP53", "TP53", "BRCA1", "BRCA1", "UNKN"),
    pathway = c("DDR",  "Other", "DDR",  "NULL",   "NULL"),
    ct      = c(5,       3,       4,       2,        1),
    stringsAsFactors = FALSE
  )
  specific <- mock_counts %>%
    filter(!pathway %in% c("NULL", "Other") & !is.na(pathway)) %>%
    group_by(gene) %>% slice_max(ct, n = 1, with_ties = FALSE) %>% ungroup()
  fallback <- mock_counts %>%
    filter(!gene %in% specific$gene) %>%
    group_by(gene) %>% slice_max(ct, n = 1, with_ties = FALSE) %>% ungroup()
  result <- bind_rows(specific, fallback)

  # TP53 and BRCA1 should get DDR (specific); UNKN gets NULL (fallback)
  stopifnot(result %>% filter(gene == "TP53")  %>% pull(pathway) == "DDR")
  stopifnot(result %>% filter(gene == "BRCA1") %>% pull(pathway) == "DDR")
  stopifnot(result %>% filter(gene == "UNKN")  %>% pull(pathway) == "NULL")
  message("\u2713 add_pathway: specific pathways prioritised over NULL/Other")
})

In [ ]:
figures <- list(
  onco   = onco_plts,
  labels = label_plts
)
saveRDS(figures, paste0(READY_FIG, "drivers.rds"))